# CardPilot MC-OCR: train, benchmark, export

Attach the Vietnamese Receipts MC-OCR dataset and a private Kaggle Dataset containing the CardPilot OCR source. This notebook keeps research outputs separate from the production service.

In [ ]:
from pathlib import Path

DATASET_ROOT = Path('/kaggle/input/vietnamese-receipts-mc-ocr-2021')
SOURCE_ROOT = Path('/kaggle/input/cardpilot-ocr-source')
WORK_ROOT = Path('/kaggle/working/cardpilot-ocr')
MODEL_VERSION = 'mcocr-cardpilot-v1'
WORK_ROOT.mkdir(parents=True, exist_ok=True)
assert DATASET_ROOT.exists(), DATASET_ROOT
assert SOURCE_ROOT.exists(), SOURCE_ROOT

package_markers = list(SOURCE_ROOT.rglob('research/data/mcocr_train_df.csv'))
if len(package_markers) != 1:
    raise RuntimeError(f'Expected one complete CardPilot OCR package, found: {package_markers}')
package_root = package_markers[0].parents[2]
print(f'Using CardPilot OCR package: {package_root}')

## 1. Inspect data and build the fixed benchmark manifest
Keep this manifest unchanged between experiments. Do not use its cases for training.

In [ ]:
images = [p for p in DATASET_ROOT.rglob('*') if p.suffix.lower() in {'.jpg', '.jpeg', '.png'}]
print(f'{len(images):,} images')
print(*images[:10], sep='\n')

In [ ]:
import shutil
import subprocess
import sys

annotations = package_root / 'research/data/mcocr_train_df.csv'
prepare = package_root / 'research/scripts/prepare_mcocr.py'
assert annotations.is_file(), annotations
assert prepare.is_file(), prepare
manifest = WORK_ROOT / 'benchmark.jsonl'
subprocess.run([
    sys.executable, str(prepare),
    '--dataset-root', str(DATASET_ROOT),
    '--annotations', str(annotations),
    '--output', str(manifest),
], check=True)

## 2. Train or fine-tune
MC-OCR is four models, not one model. Train only the component under experiment, then point the paths below at its best checkpoint. Keep the other three checkpoints at the frozen baseline. The legacy upstream commands live under `mc_ocr/rotation_corrector`, `mc_ocr/text_classifier/vietocr`, and `mc_ocr/key_info_extraction/PICK`.

In [ ]:
# Keep Kaggle's CUDA stack untouched. Install CPU Paddle in an isolated venv.
runtime_venv = WORK_ROOT / 'runtime-venv'
RUNTIME_PYTHON = runtime_venv / 'bin/python'
if not RUNTIME_PYTHON.is_file():
    subprocess.run([
        sys.executable, '-m', 'venv', '--system-site-packages', str(runtime_venv),
    ], check=True)
subprocess.run([
    str(RUNTIME_PYTHON), '-m', 'pip', 'install', '-q', '--no-cache-dir',
    'paddlepaddle==3.2.2',
    '-i', 'https://www.paddlepaddle.org.cn/packages/stable/cpu/',
], check=True)
subprocess.run([
    str(RUNTIME_PYTHON), '-c',
    'import paddle, torch; print(\"Paddle\", paddle.__version__, \"CPU; Torch\", torch.__version__, torch.version.cuda)',
], check=True)

# Download and normalize the four upstream baseline checkpoints.
if subprocess.run([sys.executable, '-c', 'import gdown']).returncode != 0:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'gdown'], check=True)

checkpoint_root = WORK_ROOT / 'checkpoints'
downloader = package_root / 'scripts/download_models.py'
assert downloader.is_file(), downloader
assert (package_root / 'mc_ocr').is_dir(), package_root
subprocess.run([
    sys.executable, str(downloader),
    '--root', str(package_root),
    '--model-dir', str(checkpoint_root),
    '--model-version', 'mc-ocr-top1-upstream',
], check=True)

# Override one of these after training when evaluating a new component.
DETECTOR = checkpoint_root / 'detector/ch_ppocr_server_v2.0_det_infer'
ROTATION = checkpoint_root / 'rotation/model.pth'
RECOGNITION = checkpoint_root / 'recognition/model.pth'
KIE = checkpoint_root / 'kie/model.pth'

# Example for a component-specific training command:
# subprocess.run(['bash', str(package_root / 'mc_ocr/key_info_extraction/PICK/dist_train.sh')], check=True)

## 3. Export the immutable artifact
The output directory is the only interface consumed by `cardpilot-ocr-service`.

In [ ]:
checkpoints = {
    'detector': DETECTOR,
    'rotation': ROTATION,
    'recognition': RECOGNITION,
    'kie': KIE,
}
missing = [f'{name}: {path}' for name, path in checkpoints.items() if not path.exists()]
if missing:
    raise FileNotFoundError('Missing OCR checkpoints:\n' + '\n'.join(missing))

exporter = next(SOURCE_ROOT.rglob('export_artifact.py'))
artifact = WORK_ROOT / 'artifacts' / MODEL_VERSION
subprocess.run([
    sys.executable, str(exporter),
    '--detector', str(DETECTOR),
    '--rotation', str(ROTATION),
    '--recognition', str(RECOGNITION),
    '--kie', str(KIE),
    '--model-version', MODEL_VERSION,
    '--output', str(artifact),
], check=True)
shutil.make_archive(str(artifact), 'zip', artifact)

## 4. Benchmark the exported artifact
Run the engine in-process against the fixed manifest. Install the service runtime dependencies in a compatible Kaggle image first. Record field accuracy plus model-stage latency. Do not compare Kaggle latency with production CPU latency.

In [ ]:
service_root = package_root
benchmark_runner = package_root / 'research/benchmark/in_process.py'
assert (service_root / 'cardpilot_service').is_dir(), service_root
assert benchmark_runner.is_file(), benchmark_runner
report = WORK_ROOT / f'{MODEL_VERSION}-report.json'
subprocess.run([
    str(RUNTIME_PYTHON), str(benchmark_runner),
    '--manifest', str(manifest),
    '--service-root', str(service_root),
    '--model-dir', str(artifact),
    '--device', 'cuda',
    '--detector-device', 'cpu',
    '--output', str(report),
], check=True)